## 1. Setup

In [7]:
import os, re, json, pickle, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import gradio as gr

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 1]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

MODEL_ICONS = {
    "SimpleRNN": "🧠",
    "LSTM": "🧠",
    "GRU": "🧠",
    "Transformer (DistilBERT)": "🤖",
}


## 2. Point at your training notebook's saved output

Edit `ARTIFACTS_ROOT` below to match wherever Kaggle mounted your training
notebook's output (see step 2/3 above). Everything else in this notebook
works automatically once this path is correct.

In [8]:
ARTIFACTS_ROOT = "/kaggle/input/notebooks/markoeed/consumer-complaint-classification"  # <-- EDIT ME

SAVED_MODELS_DIR = os.path.join(ARTIFACTS_ROOT, "saved_models")
ARTIFACTS_DIR = os.path.join(ARTIFACTS_ROOT, "artifacts")

assert os.path.exists(SAVED_MODELS_DIR), f"Not found: {SAVED_MODELS_DIR} -- check ARTIFACTS_ROOT"
assert os.path.exists(ARTIFACTS_DIR), f"Not found: {ARTIFACTS_DIR} -- check ARTIFACTS_ROOT"

print("Found saved_models:", os.listdir(SAVED_MODELS_DIR))
print("Found artifacts:", os.listdir(ARTIFACTS_DIR))


Found saved_models: ['transformer_best', 'lstm.keras', 'simplernn.keras', 'gru.keras']
Found artifacts: ['label_encoder.pkl', 'config.json', 'model_comparison.csv', 'keras_tokenizer.json']


## 3. Load config, label encoder, and the best model

In [9]:
with open(os.path.join(ARTIFACTS_DIR, "config.json")) as f:
    CONFIG = json.load(f)

BEST_MODEL_NAME = CONFIG["best_model"]
MAX_LEN = CONFIG["max_len"]
CLASS_NAMES = CONFIG["class_names"]
print("Best model (from training):", BEST_MODEL_NAME)

with open(os.path.join(ARTIFACTS_DIR, "label_encoder.pkl"), "rb") as f:
    label_encoder = pickle.load(f)

comparison_path = os.path.join(ARTIFACTS_DIR, "model_comparison.csv")
if os.path.exists(comparison_path):
    comparison_df = pd.read_csv(comparison_path, index_col=0)
else:
    comparison_df = pd.DataFrame()

def get_accuracy_for(display_name):
    if comparison_df.empty:
        return "N/A"
    for idx in comparison_df.index:
        if idx.upper() == display_name.upper():
            return f"{comparison_df.loc[idx, 'accuracy'] * 100:.1f}%"
    return "N/A"

# --- Keras RNN family: SimpleRNN, LSTM, GRU ---
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences

with open(os.path.join(ARTIFACTS_DIR, "keras_tokenizer.json")) as f:
    keras_tokenizer = tokenizer_from_json(f.read())

KERAS_MODEL_FILES = {
    "SimpleRNN": "simplernn.keras",
    "LSTM": "lstm.keras",
    "GRU": "gru.keras",
}

keras_models = {}
for display_name, filename in KERAS_MODEL_FILES.items():
    path = os.path.join(SAVED_MODELS_DIR, filename)
    if os.path.exists(path):
        keras_models[display_name] = load_model(path)
        print(f"Loaded {display_name} from {filename}")
    else:
        print(f"(skipped) {display_name} not found at {path}")

# --- Transformer ---
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

transformer_dir = os.path.join(SAVED_MODELS_DIR, "transformer_best")
hf_tokenizer = None
hf_model = None
if os.path.exists(transformer_dir):
    hf_tokenizer = AutoTokenizer.from_pretrained(transformer_dir)
    hf_model = AutoModelForSequenceClassification.from_pretrained(transformer_dir)
    hf_model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    hf_model.to(device)
    print("Loaded Transformer (DistilBERT) on:", device)
else:
    print("(skipped) Transformer not found -- only trained/saved if it was the best model")

MODEL_CHOICES = list(keras_models.keys()) + (["Transformer (DistilBERT)"] if hf_model is not None else [])
if not MODEL_CHOICES:
    raise RuntimeError("No models found under saved_models/ -- check ARTIFACTS_ROOT.")

DEFAULT_MODEL = BEST_MODEL_NAME if BEST_MODEL_NAME in MODEL_CHOICES else MODEL_CHOICES[0]
print("\nAvailable models in the app:", MODEL_CHOICES)
print("Default model:", DEFAULT_MODEL)


Best model (from training): Transformer (DistilBERT)
Loaded SimpleRNN from simplernn.keras
Loaded LSTM from lstm.keras
Loaded GRU from gru.keras


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded Transformer (DistilBERT) on: cpu

Available models in the app: ['SimpleRNN', 'LSTM', 'GRU', 'Transformer (DistilBERT)']
Default model: Transformer (DistilBERT)


## 4. Inference function

In [10]:
def run_model(cleaned_text, model_choice):
    if model_choice == "Transformer (DistilBERT)":
        inputs = hf_tokenizer(cleaned_text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
        inputs = {k: v.to(hf_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = hf_model(**inputs).logits
        return logits.softmax(dim=-1).cpu().numpy()[0]
    else:
        model = keras_models[model_choice]
        seq = keras_tokenizer.texts_to_sequences([cleaned_text])
        pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
        return model.predict(pad, verbose=0)[0]


def make_bar(fraction, width=24):
    filled = int(round(fraction * width))
    return "█" * filled + "░" * (width - filled)


def predict_complaint(text, model_choice, show_top5):
    if not text or not text.strip():
        return ("—", "0.0%", "", pd.DataFrame(),
                f"{MODEL_ICONS.get(model_choice, chr(0x1F9E0))} {model_choice}",
                get_accuracy_for(model_choice), "0.00 sec", "0")

    cleaned = clean_text(text)
    word_count = len(text.split())

    start = time.time()
    probs = run_model(cleaned, model_choice)
    elapsed = time.time() - start

    order = np.argsort(probs)[::-1]
    top_idx = int(order[0])
    top_label = CLASS_NAMES[top_idx]
    top_conf = float(probs[top_idx])
    bar = make_bar(top_conf)

    top5_rows = []
    for rank, i in enumerate(order[:5], start=1):
        label = CLASS_NAMES[int(i)].replace("_", " ").title()
        top5_rows.append([rank, label, f"{probs[int(i)] * 100:.1f}%"])
    top5_df = pd.DataFrame(top5_rows, columns=["#", "Category", "Confidence"])

    category_display = top_label.replace("_", " ").title()
    model_info = f"{MODEL_ICONS.get(model_choice, chr(0x1F9E0))} {model_choice}"
    accuracy = get_accuracy_for(model_choice)
    inference_time = f"{elapsed:.2f} sec"

    return (category_display, f"{top_conf * 100:.1f}%", bar,
            top5_df if show_top5 else pd.DataFrame(),
            model_info, accuracy, inference_time, str(word_count))


# quick sanity check
sample = "my credit card company charged me a late fee even though i paid on time"
print(predict_complaint(sample, DEFAULT_MODEL, True))


('Credit Card', '94.4%', '███████████████████████░',    #             Category Confidence
0  1          Credit Card      94.4%
1  2     Credit Reporting       4.1%
2  3      Debt Collection       1.1%
3  4       Retail Banking       0.3%
4  5  Mortgages And Loans       0.2%, '🤖 Transformer (DistilBERT)', '89.7%', '1.28 sec', '15')


## 5. Launch the app

Includes:
- **Model selector** — pick SimpleRNN / LSTM / GRU / Transformer and compare live
- **Confidence bar chart** (`gr.Label`) across all categories
- **Screen recording button** — client-side JS using `getDisplayMedia` +
  `MediaRecorder`; records your browser tab and gives you a `.webm` file to download.
  Only works on the public `https://xxxxxxxx.gradio.live` link in its own tab, not
  inside Kaggle's embedded output iframe (browser security restriction on iframes).

In [11]:
CUSTOM_CSS = """
.gradio-container {max-width: 1000px !important; margin: auto;}
#header-title {text-align: center; margin-bottom: 0.1em; font-size: 1.6em;}
#header-subtitle {text-align: center; color: #6b7280; margin-bottom: 1.4em;}
#section-label {font-weight: 700; margin-top: 1.2em; margin-bottom: 0.3em; border-bottom: 1px solid #e5e7eb; padding-bottom: 4px;}
#confidence-bar {font-family: monospace; font-size: 15px; letter-spacing: -1px; color: #2563eb;}
#category-box {font-size: 1.3em; font-weight: 700; color: #111827;}
#about-box {color: #4b5563; font-size: 0.95em; line-height: 1.5;}
footer {display: none !important;}

#rec-widget {position: fixed; bottom: 20px; right: 20px; z-index: 999;}
#rec-dots {
  background: white; border: 1px solid #e5e7eb; cursor: pointer; font-size: 20px;
  line-height: 1; padding: 8px 14px; border-radius: 50%; color: #374151;
  box-shadow: 0 2px 8px rgba(0,0,0,0.15);
}
#rec-dots:hover {background: #f3f4f6;}
#rec-menu {
  display: none; position: absolute; bottom: 46px; right: 0;
  background: white; border: 1px solid #e5e7eb; border-radius: 10px;
  box-shadow: 0 6px 20px rgba(0,0,0,0.15); padding: 14px; width: 250px; text-align: left;
}
#rec-menu.open {display: block;}
#rec-menu h4 {margin: 0 0 8px 0; font-size: 13px; color: #6b7280; text-transform: uppercase; letter-spacing: .04em;}
#rec-menu label {display: flex; align-items: center; gap: 8px; font-size: 14px; margin-bottom: 8px; color: #111827;}
#rec-menu select {width: 100%; margin-bottom: 10px; padding: 4px; border-radius: 6px; border: 1px solid #d1d5db;}
.rec-btn-row {display: flex; gap: 6px; margin-top: 4px; flex-wrap: wrap;}
.rec-btn-row button {
  flex: 1; padding: 7px 0; border-radius: 6px; border: none; cursor: pointer;
  font-weight: 600; font-size: 13px; color: white;
}
#startRec {background: #dc2626;}
#pauseRec {background: #d97706;}
#stopRec {background: #374151;}
.rec-btn-row button:disabled {opacity: 0.45; cursor: not-allowed;}
#recStatus {font-size: 12px; color: #6b7280; margin-top: 8px; text-align: center;}
#downloadLink {display: none; margin-top: 8px; font-weight: 600; color: #2563eb; text-align: center; text-decoration: none;}
"""

SCREEN_RECORD_HTML = """
<div id="rec-widget">
  <button id="rec-dots" title="Screen recording options">&#8942;</button>
  <div id="rec-menu">
    <h4>Screen Recorder</h4>
    <label><input type="checkbox" id="optAudio" checked> Include Audio</label>
    <label><input type="checkbox" id="optCursor" checked> Show Cursor</label>
    <select id="optQuality">
      <option value="high" selected>Quality: High</option>
      <option value="medium">Quality: Medium</option>
      <option value="low">Quality: Low</option>
    </select>
    <div class="rec-btn-row">
      <button id="startRec">Start</button>
      <button id="pauseRec" disabled>Pause</button>
      <button id="stopRec" disabled>Stop</button>
    </div>
    <div id="recStatus">Status: Not recording</div>
    <a id="downloadLink" download="complaint-classifier-demo.webm">Download Recording</a>
  </div>
</div>
"""

# NOTE: <script> tags inserted via gr.HTML never execute -- browsers ignore scripts
# injected through innerHTML for security reasons. All JS logic below is attached
# through Gradio's own `js=` mechanism on demo.load(...), which actually executes it.
RECORDER_JS = """
() => {
  let mediaRecorder;
  let recordedChunks = [];
  let isPaused = false;
  const QUALITY_BITRATES = {high: 8000000, medium: 3000000, low: 1000000};

  function initRecorder() {
    const dotsBtn = document.getElementById('rec-dots');
    const menu = document.getElementById('rec-menu');
    const startBtn = document.getElementById('startRec');
    const pauseBtn = document.getElementById('pauseRec');
    const stopBtn = document.getElementById('stopRec');
    const dlLink = document.getElementById('downloadLink');
    const status = document.getElementById('recStatus');
    if (!dotsBtn || dotsBtn.dataset.bound) return false;
    dotsBtn.dataset.bound = "true";

    dotsBtn.onclick = (e) => {
      e.stopPropagation();
      menu.classList.toggle('open');
    };
    document.addEventListener('click', (e) => {
      if (!menu.contains(e.target) && e.target !== dotsBtn) menu.classList.remove('open');
    });

    startBtn.onclick = async () => {
      try {
        const includeCursor = document.getElementById('optCursor').checked;
        const includeAudio = document.getElementById('optAudio').checked;
        const quality = document.getElementById('optQuality').value;

        const stream = await navigator.mediaDevices.getDisplayMedia({
          video: {cursor: includeCursor ? "always" : "never"},
          audio: includeAudio
        });
        recordedChunks = [];
        mediaRecorder = new MediaRecorder(stream, {
          mimeType: 'video/webm',
          videoBitsPerSecond: QUALITY_BITRATES[quality]
        });
        mediaRecorder.ondataavailable = e => { if (e.data.size > 0) recordedChunks.push(e.data); };
        mediaRecorder.onstop = () => {
          const blob = new Blob(recordedChunks, {type: 'video/webm'});
          const url = URL.createObjectURL(blob);
          dlLink.href = url;
          dlLink.style.display = 'block';
          status.innerText = 'Status: Recording finished';
        };
        mediaRecorder.start();
        isPaused = false;
        status.innerText = 'Status: Recording...';
        startBtn.disabled = true;
        pauseBtn.disabled = false;
        stopBtn.disabled = false;
        dlLink.style.display = 'none';
      } catch (err) {
        status.innerText = 'Status: permission denied';
        alert('Screen recording permission denied or unavailable: ' + err.message +
              '\n\nTip: open this app in its own browser tab (the public gradio.live ' +
              'link), not inside an embedded iframe.');
      }
    };

    pauseBtn.onclick = () => {
      if (!mediaRecorder) return;
      if (!isPaused) {
        mediaRecorder.pause();
        isPaused = true;
        pauseBtn.innerText = 'Resume';
        status.innerText = 'Status: Paused';
      } else {
        mediaRecorder.resume();
        isPaused = false;
        pauseBtn.innerText = 'Pause';
        status.innerText = 'Status: Recording...';
      }
    };

    stopBtn.onclick = () => {
      if (mediaRecorder && mediaRecorder.state !== 'inactive') mediaRecorder.stop();
      startBtn.disabled = false;
      pauseBtn.disabled = true;
      pauseBtn.innerText = 'Pause';
      stopBtn.disabled = true;
    };
    return true;
  }

  let attempts = 0;
  const interval = setInterval(() => {
    attempts += 1;
    if (initRecorder() || attempts > 20) clearInterval(interval);
  }, 250);
}
"""

with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate"),
               title="AI Customer Complaint Classification System") as demo:

    gr.Markdown("### 🤖 AI CUSTOMER COMPLAINT CLASSIFICATION SYSTEM", elem_id="header-title")
    gr.Markdown(
        "Automatically classify financial customer complaints using Deep Learning and Transformer models.",
        elem_id="header-subtitle",
    )

    gr.Markdown("Complaint", elem_id="section-label")
    text_input = gr.Textbox(
        lines=6,
        show_label=False,
        placeholder="Type or paste a customer complaint here...",
    )

    with gr.Row():
        model_dropdown = gr.Dropdown(choices=MODEL_CHOICES, value=DEFAULT_MODEL, label="Model")
        show_top5 = gr.Checkbox(value=True, label="Show Top 5 Predictions")

    with gr.Row():
        predict_btn = gr.Button("Predict", variant="primary")
        clear_btn = gr.Button("Clear")

    gr.Markdown("Prediction", elem_id="section-label")
    with gr.Row():
        with gr.Column():
            gr.Markdown("🏷 Category")
            category_out = gr.Markdown("—", elem_id="category-box")
        with gr.Column():
            gr.Markdown("📈 Confidence")
            confidence_out = gr.Markdown("0.0%")
            bar_out = gr.Markdown("", elem_id="confidence-bar")

    top5_out = gr.Dataframe(
        headers=["#", "Category", "Confidence"],
        label="Top Predictions",
        interactive=False,
        row_count=5,
    )

    gr.Markdown("Model Information", elem_id="section-label")
    with gr.Row():
        model_info_out = gr.Markdown(f"{MODEL_ICONS.get(DEFAULT_MODEL, chr(0x1F9E0))} {DEFAULT_MODEL}")
        with gr.Column():
            gr.Markdown("Accuracy")
            accuracy_out = gr.Markdown(get_accuracy_for(DEFAULT_MODEL))
        with gr.Column():
            gr.Markdown("Inference Time")
            time_out = gr.Markdown("0.00 sec")
        with gr.Column():
            gr.Markdown("Input Words")
            words_out = gr.Markdown("0")

    gr.Markdown("About", elem_id="section-label")
    gr.Markdown(
        "This application uses Natural Language Processing to automatically classify "
        "customer complaints into their appropriate CFPB categories, helping financial "
        "institutions streamline complaint management.",
        elem_id="about-box",
    )

    gr.HTML(SCREEN_RECORD_HTML)

    outputs = [category_out, confidence_out, bar_out, top5_out,
               model_info_out, accuracy_out, time_out, words_out]

    predict_btn.click(fn=predict_complaint, inputs=[text_input, model_dropdown, show_top5], outputs=outputs)
    text_input.submit(fn=predict_complaint, inputs=[text_input, model_dropdown, show_top5], outputs=outputs)
    clear_btn.click(
        fn=lambda: ("", "—", "0.0%", "", pd.DataFrame(),
                    f"{MODEL_ICONS.get(DEFAULT_MODEL, chr(0x1F9E0))} {DEFAULT_MODEL}",
                    get_accuracy_for(DEFAULT_MODEL), "0.00 sec", "0"),
        inputs=None,
        outputs=[text_input] + outputs,
    )

    demo.load(fn=None, inputs=None, outputs=None, js=RECORDER_JS)

demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9543f7e89cb2d25c36.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
